# 21. RL for diffusion generation — DDPO with real DDPM reverse transitions

The denoising trajectory is treated as an MDP exactly as in DDPO, but the previous arbitrary `x + dt * f(x,t)` Gaussian transition has been removed. This version derives each stochastic reverse-policy transition from a discrete DDPM schedule and the denoiser's predicted noise.

Only latent dimension, batch size, denoiser width, and number of diffusion steps are reduced.


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(4)
device = torch.device("cpu")
print("device:", device)


## 1. Small epsilon-prediction denoiser

The network is deliberately small, but the policy transition is the DDPM posterior induced by its `epsilon_theta(x_t,t)` prediction.


In [ ]:
class EpsilonDenoiser(nn.Module):
    def __init__(self, latent_dim=2, hidden_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def forward(self, x_t, normalized_t):
        model_input = torch.cat(
            [x_t, normalized_t[:, None]],
            dim=-1,
        )
        return self.net(model_input)


## 2. DDPM schedule and reverse posterior

For a discrete schedule, the model predicts epsilon, reconstructs `x0_hat`, and inserts it into the exact Gaussian posterior `q(x_{t-1}|x_t,x0_hat)`. That posterior mean and variance define the stochastic policy used by DDPO.


In [ ]:
betas = torch.linspace(0.05, 0.12, 5, device=device)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)


def ddpm_reverse_mean_sigma(model, x_t, time_index):
    batch_size = x_t.size(0)
    normalized_t = torch.full(
        (batch_size,),
        time_index / (len(betas) - 1),
        device=x_t.device,
    )

    predicted_epsilon = model(x_t, normalized_t)

    alpha_t = alphas[time_index]
    beta_t = betas[time_index]
    alpha_bar_t = alpha_bars[time_index]
    alpha_bar_previous = alpha_bars[time_index - 1]

    x0_prediction = (
        x_t
        - torch.sqrt(1 - alpha_bar_t) * predicted_epsilon
    ) / torch.sqrt(alpha_bar_t)

    x0_coefficient = (
        torch.sqrt(alpha_bar_previous)
        * beta_t
        / (1 - alpha_bar_t)
    )
    xt_coefficient = (
        torch.sqrt(alpha_t)
        * (1 - alpha_bar_previous)
        / (1 - alpha_bar_t)
    )
    posterior_mean = (
        x0_coefficient * x0_prediction
        + xt_coefficient * x_t
    )

    posterior_variance = (
        beta_t
        * (1 - alpha_bar_previous)
        / (1 - alpha_bar_t)
    )
    posterior_sigma = torch.sqrt(
        posterior_variance.clamp_min(1e-6)
    )
    return posterior_mean, posterior_sigma


def gaussian_log_probability(sample, mean, sigma):
    elementwise = (
        -0.5 * ((sample - mean) / sigma).square()
        - torch.log(sigma)
        - 0.5 * math.log(2 * math.pi)
    )
    return elementwise.sum(dim=-1)


## 3. Roll out the complete old-policy trajectory

The state, sampled next latent, timestep, and old transition log-probability are stored at every stochastic reverse step.


In [ ]:
old_policy = EpsilonDenoiser().to(device)
current_policy = EpsilonDenoiser().to(device)
current_policy.load_state_dict(old_policy.state_dict())

batch_size = 12
x_t = torch.randn(batch_size, 2, device=device)

states = []
next_states = []
time_indices = []
old_log_probs = []

with torch.no_grad():
    for time_index in range(len(betas) - 1, 0, -1):
        mean, sigma = ddpm_reverse_mean_sigma(
            old_policy,
            x_t,
            time_index,
        )
        x_next = mean + sigma * torch.randn_like(x_t)
        log_prob = gaussian_log_probability(
            x_next,
            mean,
            sigma,
        )

        states.append(x_t.clone())
        next_states.append(x_next.clone())
        time_indices.append(time_index)
        old_log_probs.append(log_prob.clone())
        x_t = x_next

final_samples = x_t
old_log_probs = torch.stack(old_log_probs, dim=1)
print("trajectory transitions:", old_log_probs.shape)


## 4. Terminal reward and fixed rollout advantage


In [ ]:
target = torch.zeros(2, device=device)
reward = -(final_samples - target).square().sum(dim=-1)
advantage = (
    reward - reward.mean()
) / (reward.std(unbiased=False) + 1e-6)

print("reward mean:", reward.mean().item())
print("advantage mean:", advantage.mean().item())


## 5. Re-evaluate every sampled transition under the current policy


In [ ]:
def reevaluate_trajectory(model):
    new_log_probs = []

    for transition_index, time_index in enumerate(time_indices):
        state_t = states[transition_index]
        sampled_next = next_states[transition_index]

        mean, sigma = ddpm_reverse_mean_sigma(
            model,
            state_t,
            time_index,
        )
        new_log_prob = gaussian_log_probability(
            sampled_next,
            mean,
            sigma,
        )
        new_log_probs.append(new_log_prob)

    return torch.stack(new_log_probs, dim=1)


## 6. Five DDPO/PPO optimization steps on the fixed rollout

This is a computation sanity check, not a quality experiment. The same old-policy rollout is reused for five optimizer steps so the clipped objective can be inspected directly.


In [ ]:
optimizer = torch.optim.AdamW(
    current_policy.parameters(),
    lr=1e-3,
)

loss_history = []
for step in range(5):
    optimizer.zero_grad()

    new_log_probs = reevaluate_trajectory(current_policy)
    ratio = torch.exp(
        new_log_probs - old_log_probs.detach()
    )

    trajectory_advantage = advantage[:, None]
    unclipped = ratio * trajectory_advantage
    clipped = ratio.clamp(0.8, 1.2) * trajectory_advantage
    loss = -torch.minimum(unclipped, clipped).mean()

    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())
    print(f"step {step + 1}: DDPO loss={loss.item():.8f}")

print("loss history:", loss_history)


## 7. Deterministic flow models are a different policy construction

A deterministic flow ODE still has no Gaussian transition probability by itself. A flow-RL method must derive a suitable stochastic policy or its own objective; the DDPM transition ratio above is not copied onto deterministic flow matching.


## References and provenance

- DDPM: epsilon prediction and the Gaussian reverse posterior.
- DDPO: the denoising process as a multi-step MDP, saved old-policy transition log-probabilities, trajectory rewards, and PPO-style clipped importance ratios.

Only tensor dimensions, batch size, and diffusion-step count are reduced.
